# 01 — Environment Setup

Imports, device detection, and path configuration.  
Run this notebook first; the cells below set variables (`DATA_DIR`, `CLUSTER_DIR`, `DEVICE`) used by the training and evaluation notebooks.

In [ ]:
import sys, os
sys.path.insert(0, '..')  # make src/ importable

import warnings, random
import numpy as np
import torch
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

from src.config import DEVICE
print('PyTorch :', torch.__version__)
print('Device  :', DEVICE)

In [ ]:
import zipfile, shutil

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    ZIP_PATH_IN_DRIVE    = '/content/drive/MyDrive/good_data.zip'
    CLUSTER_DIR_IN_DRIVE = '/content/drive/MyDrive/clustering_v2'
    LOCAL_DATA_ROOT      = '/content'
    LOCAL_DATA_DIR       = os.path.join(LOCAL_DATA_ROOT, 'good_data')

    assert os.path.exists(ZIP_PATH_IN_DRIVE), (
        f'Could not find {ZIP_PATH_IN_DRIVE}. '
        f'Update ZIP_PATH_IN_DRIVE to point at good_data.zip in your Drive.'
    )

    already_extracted = os.path.isdir(LOCAL_DATA_DIR) and len(os.listdir(LOCAL_DATA_DIR)) > 0
    if not already_extracted:
        print(f'Extracting {ZIP_PATH_IN_DRIVE} -> {LOCAL_DATA_ROOT} ...')
        with zipfile.ZipFile(ZIP_PATH_IN_DRIVE, 'r') as zf:
            zf.extractall(LOCAL_DATA_ROOT)

    # Flatten nested good_data/good_data if present
    inner = os.path.join(LOCAL_DATA_DIR, 'good_data')
    if os.path.isdir(inner):
        for item in os.listdir(inner):
            shutil.move(os.path.join(inner, item), os.path.join(LOCAL_DATA_DIR, item))
        os.rmdir(inner)

    DATA_DIR    = LOCAL_DATA_DIR
    CLUSTER_DIR = CLUSTER_DIR_IN_DRIVE
    assert os.path.isdir(CLUSTER_DIR), (
        f'Could not find {CLUSTER_DIR}. Train the IL model first or update CLUSTER_DIR_IN_DRIVE.'
    )
else:
    # Local fallback — edit these paths for your machine
    DATA_DIR    = '../good_data'
    CLUSTER_DIR = '../clustering_v2'

print(f'DATA_DIR    = {DATA_DIR}')
print(f'CLUSTER_DIR = {CLUSTER_DIR}')

In [ ]:
# Checkpoint and log paths — derived from CLUSTER_DIR
IL_WEIGHTS        = os.path.join(CLUSTER_DIR, 'transformer_imitation_v2.pt')
RL_SAVE           = os.path.join(CLUSTER_DIR, 'transformer_rl_v2_K.pt')
RL_BASELINE_CACHE = os.path.join(CLUSTER_DIR, 'il_baseline_cache_K.pkl')
RL_LOG            = os.path.join(CLUSTER_DIR, 'rl_training_log_v2_K.csv')
OUTPUT_DIR        = os.path.join(CLUSTER_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

for p in [IL_WEIGHTS, DATA_DIR]:
    print(f"{'OK' if os.path.exists(p) else 'MISSING'}  {p}")